# Note 02 Deep Dive: Gaussian Elimination, LU, Pivoting and Rank

Companion notebook for [`02-Gaussian-Elimination-Row-Operations-Rank.md`](../02-Gaussian-Elimination-Row-Operations-Rank.md).

| Part | Topic |
|---|---|
| A | Step-by-step elimination printer (exact fractions): lecture example, Gauss–Jordan with a row swap, the 5×3 REF |
| B | LU factorisation from scratch vs `scipy.linalg.lu`; solving two right-hand sides with one factorisation |
| C | Why partial pivoting matters: the 1e-20 pivot |
| D | Timing and accuracy: `solve` (LU) vs explicit inverse |
| E | Rank facts checked numerically; the Rouché–Capelli classifier; parametric solutions |
| F | Applications: Kirchhoff circuit, traffic flow, balancing chemical equations, the dummy-variable trap |
| G | Rounding and growth: 3-significant-digit elimination, Wilkinson's growth matrix, rank over GF(2) |
| H | Checking the practice-problem answers of the note |

In [1]:
from fractions import Fraction
import time
import numpy as np
import scipy.linalg as sl
import sympy as sp
np.set_printoptions(precision=4, suppress=True)

## Part A — A step-by-step elimination printer

Exact `Fraction` arithmetic, so the printed matrices match hand working. `reduced=True` gives Gauss–Jordan (RREF); `reduced=False` stops at REF.

In [2]:
def show(M, msg):
    print(msg)
    w = max(len(str(x)) for r in M for x in r)
    for r in M:
        print("   [" + "  ".join(str(x).rjust(w) for x in r) + "]")

def eliminate(M, n_coef=None, reduced=True, partial=False, verbose=True):
    """Row-reduce M (list of lists). n_coef = number of coefficient columns (rest = augmented part).
    Returns (result, pivot_columns, number_of_swaps)."""
    M = [[Fraction(x) for x in r] for r in M]
    m, n = len(M), len(M[0]); n_coef = n if n_coef is None else n_coef
    r, pivots, swaps = 0, [], 0
    for c in range(n_coef):
        if r == m:
            break
        cand = [i for i in range(r, m) if M[i][c] != 0]
        if not cand:
            continue                                    # no pivot here -> free column
        p = max(cand, key=lambda i: abs(M[i][c])) if partial else cand[0]
        if p != r:
            M[r], M[p] = M[p], M[r]; swaps += 1
            verbose and show(M, f"R{r+1} <-> R{p+1}")
        if reduced and M[r][c] != 1:
            k = M[r][c]; M[r] = [x / k for x in M[r]]
            verbose and show(M, f"R{r+1} <- R{r+1} / ({k})")
        for i in range(m):
            if i == r or M[i][c] == 0 or (not reduced and i < r):
                continue
            f = M[i][c] / M[r][c]
            M[i] = [a - f * b for a, b in zip(M[i], M[r])]
            verbose and show(M, f"R{i+1} <- R{i+1} - ({f}) R{r+1}")
        pivots.append(c); r += 1
    return M, pivots, swaps

In [3]:
# Lecture example (notes p.13): x1+x2+x3=3, x1-x2+x3=1, x1+x2-x3=1
_ = eliminate([[1, 1, 1, 3], [1, -1, 1, 1], [1, 1, -1, 1]], n_coef=3, reduced=False)

R2 <- R2 - (1) R1
   [ 1   1   1   3]
   [ 0  -2   0  -2]
   [ 1   1  -1   1]
R3 <- R3 - (1) R1
   [ 1   1   1   3]
   [ 0  -2   0  -2]
   [ 0   0  -2  -2]


In [4]:
# Worked Example 1 of the note: Gauss–Jordan with a forced row swap (a11 = 0)
R, piv, swaps = eliminate([[0, 2, 1, 1], [1, -2, -3, -5], [-1, 1, 2, 3]], n_coef=3)
print("pivot columns:", piv, "| swaps:", swaps)

R1 <-> R2
   [ 1  -2  -3  -5]
   [ 0   2   1   1]
   [-1   1   2   3]
R3 <- R3 - (-1) R1
   [ 1  -2  -3  -5]
   [ 0   2   1   1]
   [ 0  -1  -1  -2]
R2 <- R2 / (2)
   [  1   -2   -3   -5]
   [  0    1  1/2  1/2]
   [  0   -1   -1   -2]
R1 <- R1 - (-2) R2
   [  1    0   -2   -4]
   [  0    1  1/2  1/2]
   [  0   -1   -1   -2]
R3 <- R3 - (-1) R2
   [   1     0    -2    -4]
   [   0     1   1/2   1/2]
   [   0     0  -1/2  -3/2]
R3 <- R3 / (-1/2)
   [  1    0   -2   -4]
   [  0    1  1/2  1/2]
   [  0    0    1    3]
R1 <- R1 - (-2) R3
   [  1    0    0    2]
   [  0    1  1/2  1/2]
   [  0    0    1    3]
R2 <- R2 - (1/2) R3
   [ 1   0   0   2]
   [ 0   1   0  -1]
   [ 0   0   1   3]
pivot columns: [0, 1, 2] | swaps: 1


In [5]:
# The 5x3 REF example (notes pp. 16-18) and its RREF
A53 = [[1, 1, 1], [1, 1, 1], [1, -1, 1], [1, 1, -1], [-1, 1, -1]]
REF, piv, _ = eliminate(A53, reduced=False, verbose=False); show(REF, "REF (our own pivot order)")
RREF, piv, _ = eliminate(A53, verbose=False); show(RREF, "RREF"); print("rank =", len(piv))
print("SymPy agrees:", sp.Matrix(A53).rref())

REF (our own pivot order)
   [ 1   1   1]
   [ 0  -2   0]
   [ 0   0  -2]
   [ 0   0   0]
   [ 0   0   0]
RREF
   [1  0  0]
   [0  1  0]
   [0  0  1]
   [0  0  0]
   [0  0  0]
rank = 3
SymPy agrees: (Matrix([
[1, 0, 0],
[0, 1, 0],
[0, 0, 1],
[0, 0, 0],
[0, 0, 0]]), (0, 1, 2))


## Part B — LU factorisation

Elimination without row swaps records the multipliers $`l_{ij}`$ in a unit lower-triangular $`L`$; what is left is $`U`$.

In [6]:
def lu_nopivot(A):
    A = np.array(A, dtype=float); n = len(A)
    L, U = np.eye(n), A.copy()
    for k in range(n - 1):
        if U[k, k] == 0:
            raise ZeroDivisionError(f"zero pivot at step {k+1}: need a row swap")
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]          # the multiplier
            U[i, k:] -= L[i, k] * U[k, k:]
    return L, U

def forward(L, b):
    y = np.zeros(len(b))
    for i in range(len(b)):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

def backward(U, y):
    n = len(y); x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

A = np.array([[2, 1, 1], [4, -6, 0], [-2, 7, 2]], dtype=float)
L, U = lu_nopivot(A)
print("L =\n", L); print("U =\n", U); print("L @ U == A:", np.allclose(L @ U, A))
print("det A = prod(diag U) =", np.prod(np.diag(U)), "| numpy:", round(np.linalg.det(A), 6))
for b in ([5, -2, 9], [1, 2, 3]):
    y = forward(L, np.array(b, float)); x = backward(U, y)
    print(f"b = {b}:  y = {y},  x = {x}")

L =
 [[ 1.  0.  0.]
 [ 2.  1.  0.]
 [-1. -1.  1.]]
U =
 [[ 2.  1.  1.]
 [ 0. -8. -2.]
 [ 0.  0.  1.]]
L @ U == A: True
det A = prod(diag U) = -16.0 | numpy: -16.0
b = [5, -2, 9]:  y = [  5. -12.   2.],  x = [1. 1. 2.]
b = [1, 2, 3]:  y = [1. 0. 4.],  x = [-1. -1.  4.]


In [7]:
# SciPy's convention: A = P @ L @ U, with partial pivoting (largest |pivot| in each column)
P, Ls, Us = sl.lu(A)
print("P =\n", P); print("L =\n", Ls); print("U =\n", Us)
print("P @ L @ U == A:", np.allclose(P @ Ls @ Us, A))
print("all |multipliers| <= 1:", np.all(np.abs(Ls) <= 1 + 1e-12))
# Factor once, solve many: lu_factor / lu_solve (what LAPACK getrf / getrs do)
lu, piv = sl.lu_factor(A)
for b in ([5, -2, 9], [1, 2, 3]):
    print(b, "->", sl.lu_solve((lu, piv), b))

P =
 [[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
L =
 [[ 1.   0.   0. ]
 [ 0.5  1.   0. ]
 [-0.5  1.   1. ]]
U =
 [[ 4. -6.  0.]
 [ 0.  4.  1.]
 [ 0.  0.  1.]]
P @ L @ U == A: True
all |multipliers| <= 1: True
[5, -2, 9] -> [1. 1. 2.]
[1, 2, 3] -> [-1. -1.  4.]


## Part C — Partial pivoting: the 1e-20 pivot

In [8]:
def ge_solve(A, b, pivot=True):
    """Plain Gaussian elimination + back substitution in float64."""
    A = np.array(A, float); b = np.array(b, float); n = len(b)
    for k in range(n - 1):
        if pivot:
            p = k + np.argmax(np.abs(A[k:, k]))
            A[[k, p]] = A[[p, k]]; b[[k, p]] = b[[p, k]]
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]; b[i] -= m * b[k]
    return backward(A, b)

eps = 1e-20
A_bad = [[eps, 1], [1, 1]]; b_bad = [1, 2]
print("exact (to 16 digits):", [1 / (1 - eps), (1 - 2 * eps) / (1 - eps)])
print("no pivoting       :", ge_solve(A_bad, b_bad, pivot=False))
print("partial pivoting  :", ge_solve(A_bad, b_bad, pivot=True))
print("numpy.linalg.solve:", np.linalg.solve(A_bad, b_bad))

exact (to 16 digits): [1.0, 1.0]
no pivoting       : [0. 1.]
partial pivoting  : [1. 1.]
numpy.linalg.solve: [1. 1.]


In [9]:
# How the error grows as the pivot shrinks (this is figure images/02x_pivoting_error.png)
for k in [2, 5, 8, 11, 14, 17, 20]:
    e = 10.0 ** -k
    x_np = ge_solve([[e, 1], [1, 1]], [1, 2], pivot=False)
    x_p = ge_solve([[e, 1], [1, 1]], [1, 2], pivot=True)
    print(f"eps=1e-{k:<2d}  no-pivot x1 = {x_np[0]:.16f}   pivot x1 = {x_p[0]:.16f}")

eps=1e-2   no-pivot x1 = 1.0101010101010055   pivot x1 = 1.0101010101010099
eps=1e-5   no-pivot x1 = 1.0000100000961964   pivot x1 = 1.0000100001000010
eps=1e-8   no-pivot x1 = 1.0000000050247593   pivot x1 = 1.0000000099999999
eps=1e-11  no-pivot x1 = 1.0000000827403710   pivot x1 = 1.0000000000100000
eps=1e-14  no-pivot x1 = 0.9992007221626409   pivot x1 = 1.0000000000000100
eps=1e-17  no-pivot x1 = 0.0000000000000000   pivot x1 = 1.0000000000000000
eps=1e-20  no-pivot x1 = 0.0000000000000000   pivot x1 = 1.0000000000000000


## Part D — Elimination vs explicit inverse: time and accuracy

In [10]:
rng = np.random.default_rng(0)
for n in [100, 400, 1600]:
    A = rng.standard_normal((n, n)); b = rng.standard_normal(n)
    np.linalg.solve(A, b); np.linalg.inv(A)              # warm-up
    reps = 20 if n <= 400 else 5
    t0 = time.perf_counter(); [np.linalg.solve(A, b) for _ in range(reps)]; ts = (time.perf_counter() - t0) / reps
    t0 = time.perf_counter(); [np.linalg.inv(A) @ b for _ in range(reps)]; ti = (time.perf_counter() - t0) / reps
    x1 = np.linalg.solve(A, b); x2 = np.linalg.inv(A) @ b
    r1 = np.linalg.norm(A @ x1 - b); r2 = np.linalg.norm(A @ x2 - b)
    print(f"n={n:5d}  solve {ts*1e3:8.2f} ms  inv@b {ti*1e3:8.2f} ms  (x{ti/ts:.1f})   residual solve {r1:.1e}  inv {r2:.1e}")

n=  100  solve     0.11 ms  inv@b     0.18 ms  (x1.6)   residual solve 4.2e-13  inv 3.5e-13
n=  400  solve     1.01 ms  inv@b     3.23 ms  (x3.2)   residual solve 3.0e-12  inv 4.7e-12


n= 1600  solve    43.06 ms  inv@b   167.98 ms  (x3.9)   residual solve 6.1e-11  inv 4.5e-11


In [11]:
# Flop counts: elimination ~ 2n^3/3, each triangular solve ~ n^2, inverse ~ 2n^3
for n in [3, 10, 100, 1000]:
    print(f"n={n:5d}: LU {2*n**3/3:>14,.0f}   2 triangular solves {2*n**2:>10,}   inverse {2*n**3:>16,}")

n=    3: LU             18   2 triangular solves         18   inverse               54
n=   10: LU            667   2 triangular solves        200   inverse            2,000
n=  100: LU        666,667   2 triangular solves     20,000   inverse        2,000,000
n= 1000: LU    666,666,667   2 triangular solves  2,000,000   inverse    2,000,000,000


## Part E — Rank facts and the Rouché–Capelli classifier

In [12]:
def classify(A, b):
    A = sp.Matrix(A); Ab = A.row_join(sp.Matrix(b))
    rA, rAb, n = A.rank(), Ab.rank(), A.cols
    if rA < rAb:
        return f"rank A = {rA} < rank [A|b] = {rAb}: INCONSISTENT (no solution)"
    if rA == n:
        return f"rank A = rank [A|b] = {rA} = n: UNIQUE solution"
    return f"rank A = rank [A|b] = {rA} < n = {n}: INFINITELY many ({n - rA} free variable(s))"

A3 = [[1, 2, -1], [2, 5, 1], [1, 3, 2]]
print(classify(A3, [3, 8, 5])); print(classify(A3, [3, 8, 6])); print(classify([[1, 1, 1], [1, -1, 1], [1, 1, -1]], [3, 1, 1]))
x, y, z = sp.symbols("x y z")
print("parametric solution:", sp.linsolve((sp.Matrix(A3), sp.Matrix([3, 8, 5])), x, y, z))

rank A = rank [A|b] = 2 < n = 3: INFINITELY many (1 free variable(s))
rank A = 2 < rank [A|b] = 3: INCONSISTENT (no solution)
rank A = rank [A|b] = 3 = n: UNIQUE solution
parametric solution: {(7*z - 1, 2 - 3*z, z)}


In [13]:
# 3 equations, 4 unknowns: two free variables
A34 = sp.Matrix([[1, 3, 0, 2], [0, 0, 1, 4], [1, 3, 1, 6]]); b34 = sp.Matrix([1, 6, 7])
print(A34.row_join(b34).rref())
print(sp.linsolve((A34, b34), *sp.symbols("x1:5")))

(Matrix([
[1, 3, 0, 2, 1],
[0, 0, 1, 4, 6],
[0, 0, 0, 0, 0]]), (0, 2))
{(-3*x2 - 2*x4 + 1, x2, 6 - 4*x4, x4)}


In [14]:
# Rank examples worked in the note
for M in ([[1, 2, 3], [4, 5, 6], [7, 8, 9]],
          [[1, 1, 0, 2], [2, 3, 1, 4], [0, 1, 1, 1], [3, 4, 1, 7]],
          [[2, 4, 1, 3], [1, 2, 1, 1], [3, 6, 2, 4], [1, 2, 0, 2]]):
    M = np.array(M); print(M.shape, "rank", np.linalg.matrix_rank(M), "| rank of transpose", np.linalg.matrix_rank(M.T))

(3, 3) rank 2 | rank of transpose 2
(4, 4) rank 3 | rank of transpose 3
(4, 4) rank 2 | rank of transpose 2


In [15]:
# rank(AB) <= min(rank A, rank B) and rank(A^T A) = rank(A), on random low-rank matrices
rng = np.random.default_rng(1)
for trial in range(5):
    A = rng.standard_normal((6, 2)) @ rng.standard_normal((2, 5))     # rank 2, 6x5
    B = rng.standard_normal((5, 3)) @ rng.standard_normal((3, 4))     # rank 3, 5x4
    r = np.linalg.matrix_rank
    print(f"rank A={r(A)}, rank B={r(B)}, rank AB={r(A @ B)}, rank AtA={r(A.T @ A)}, rank AAt={r(A @ A.T)}")

rank A=2, rank B=3, rank AB=2, rank AtA=2, rank AAt=2
rank A=2, rank B=3, rank AB=2, rank AtA=2, rank AAt=2
rank A=2, rank B=3, rank AB=2, rank AtA=2, rank AAt=2
rank A=2, rank B=3, rank AB=2, rank AtA=2, rank AAt=2
rank A=2, rank B=3, rank AB=2, rank AtA=2, rank AAt=2


## Part F — Applications

In [16]:
# Kirchhoff circuit: E1 = 4 V with R1 = 2 ohm (left), R2 = 1 ohm (middle), E2 = 6 V with R3 = 4 ohm (right)
# node:  I1 - I2 + I3 = 0 ;  left loop: 2 I1 + 1 I2 = 4 ;  right loop: 1 I2 + 4 I3 = 6
K = np.array([[1, -1, 1], [2, 1, 0], [0, 1, 4]], float); e = np.array([0, 4, 6], float)
print("currents (A):", np.linalg.solve(K, e))
_ = eliminate([[1, -1, 1, 0], [2, 1, 0, 4], [0, 1, 4, 6]], n_coef=3)

currents (A): [1. 2. 1.]
R2 <- R2 - (2) R1
   [ 1  -1   1   0]
   [ 0   3  -2   4]
   [ 0   1   4   6]
R2 <- R2 / (3)
   [   1    -1     1     0]
   [   0     1  -2/3   4/3]
   [   0     1     4     6]
R1 <- R1 - (-1) R2
   [   1     0   1/3   4/3]
   [   0     1  -2/3   4/3]
   [   0     1     4     6]
R3 <- R3 - (1) R2
   [   1     0   1/3   4/3]
   [   0     1  -2/3   4/3]
   [   0     0  14/3  14/3]
R3 <- R3 / (14/3)
   [   1     0   1/3   4/3]
   [   0     1  -2/3   4/3]
   [   0     0     1     1]
R1 <- R1 - (1/3) R3
   [   1     0     0     1]
   [   0     1  -2/3   4/3]
   [   0     0     1     1]
R2 <- R2 - (-2/3) R3
   [1  0  0  1]
   [0  1  0  2]
   [0  0  1  1]


In [17]:
# Traffic flow around a one-way block A -> B -> C -> D -> A (x1 = AB, x2 = BC, x3 = CD, x4 = DA)
# Conservation at each intersection: flow in = flow out (vehicles per hour)
# A: x4 + 500 = x1 + 200   B: x1 + 100 = x2 + 300   C: x2 + 400 = x3 + 300   D: x3 + 100 = x4 + 300
T = sp.Matrix([[1, 0, 0, -1], [1, -1, 0, 0], [0, 1, -1, 0], [0, 0, 1, -1]]); t = sp.Matrix([300, 200, -100, 200])
print("rank T =", T.rank(), "| rank [T|t] =", T.row_join(t).rank())
print("RREF:", T.row_join(t).rref()[0].tolist())
x1, x2, x3, x4 = sp.symbols("x1:5")
sol = list(sp.linsolve((T, t), x1, x2, x3, x4))[0]
print("solution:", sol)
print("smallest non-negative flows (x4 = 0):", [v.subs(x4, 0) for v in sol])

rank T = 3 | rank [T|t] = 3
RREF: [[1, 0, 0, -1, 300], [0, 1, 0, -1, 100], [0, 0, 1, -1, 200], [0, 0, 0, 0, 0]]
solution: (x4 + 300, x4 + 100, x4 + 200, x4)
smallest non-negative flows (x4 = 0): [300, 100, 200, 0]


In [18]:
# Balancing chemical equations = finding an integer vector in the null space (one row per element)
def balance(E, reactants, products):
    v = sp.Matrix(E).nullspace()
    assert len(v) == 1, "null space must be 1-dimensional"
    v = v[0] * sp.ilcm(*[sp.fraction(t)[1] for t in v[0]])
    names = reactants + products; r = len(reactants)
    side = lambda idx: " + ".join(f"{v[i]} {names[i]}" for i in idx)
    print(side(range(r)), "->", side(range(r, len(names))), f"   (rank {sp.Matrix(E).rank()}, {len(names)} unknowns)")

#            C3H8 O2 CO2 H2O   (products entered with a minus sign)
balance([[3, 0, -1, 0],    # C
         [8, 0, 0, -2],    # H
         [0, 2, -2, -1]],  # O
        ["C3H8", "O2"], ["CO2", "H2O"])
#            CO2 H2O C6H12O6 O2   (photosynthesis)
balance([[1, 0, -6, 0], [0, 2, -12, 0], [2, 1, -6, -2]], ["CO2", "H2O"], ["C6H12O6", "O2"])
#            Al O2 Al2O3
balance([[1, 0, -2], [0, 2, -3]], ["Al", "O2"], ["Al2O3"])

1 C3H8 + 5 O2 -> 3 CO2 + 4 H2O    (rank 3, 4 unknowns)
6 CO2 + 6 H2O -> 1 C6H12O6 + 6 O2    (rank 3, 4 unknowns)
4 Al + 3 O2 -> 2 Al2O3    (rank 2, 3 unknowns)


In [19]:
# The dummy-variable trap: intercept + one-hot columns for ALL categories -> rank deficient design matrix
import pandas as pd
city = pd.Series(["Pune", "Delhi", "Chennai", "Pune", "Delhi", "Chennai"])
X_full = np.column_stack([np.ones(6), pd.get_dummies(city).to_numpy(float)])
X_drop = np.column_stack([np.ones(6), pd.get_dummies(city, drop_first=True).to_numpy(float)])
for name, X in [("intercept + all 3 dummies", X_full), ("intercept + 2 dummies (drop_first)", X_drop)]:
    print(f"{name:36s} shape {X.shape}  rank X = {np.linalg.matrix_rank(X)}  rank XtX = {np.linalg.matrix_rank(X.T @ X)}")
print("null vector of X_full:", X_full @ np.array([1, -1, -1, -1]))

intercept + all 3 dummies            shape (6, 4)  rank X = 3  rank XtX = 3
intercept + 2 dummies (drop_first)   shape (6, 3)  rank X = 3  rank XtX = 3
null vector of X_full: [0. 0. 0. 0. 0. 0.]


## Part G — Rounding, growth and other fields

In [20]:
# Elimination in 3-significant-digit arithmetic: 0.0001 x + y = 1,  x + y = 2
r3 = lambda v: float(f"{v:.3g}")
m = r3(1 / 0.0001); a22 = r3(1 - m * 1); b2 = r3(2 - m * 1); y = r3(b2 / a22); x = r3((1 - y) / 0.0001)
print(f"no pivoting : m = {m}, a22 = {a22}, b2 = {b2}  ->  x = {x}, y = {y}")
m = r3(0.0001); a22 = r3(1 - m * 1); b2 = r3(1 - m * 2); y = r3(b2 / a22); x = r3(2 - y)
print(f"pivoting    : m = {m}, a22 = {a22}, b2 = {b2}  ->  x = {x}, y = {y}")
print("exact       :", np.linalg.solve([[1e-4, 1], [1, 1]], [1, 2]))

no pivoting : m = 10000.0, a22 = -10000.0, b2 = -10000.0  ->  x = 0.0, y = 1.0
pivoting    : m = 0.0001, a22 = 1.0, b2 = 1.0  ->  x = 1.0, y = 1.0
exact       : [1.0001 0.9999]


In [21]:
# Wilkinson's matrix: partial pivoting still lets entries grow like 2^(n-1) (worst case, rare in practice)
for n in [5, 10, 20]:
    W = np.tril(-np.ones((n, n)), -1) + np.eye(n); W[:, -1] = 1
    P, L, U = sl.lu(W)
    print(f"n={n:2d}: max |A| = {np.abs(W).max():.0f},  max |U| = {np.abs(U).max():.0f}  (2^(n-1) = {2**(n-1)})")

n= 5: max |A| = 1,  max |U| = 16  (2^(n-1) = 16)
n=10: max |A| = 1,  max |U| = 512  (2^(n-1) = 512)
n=20: max |A| = 1,  max |U| = 524288  (2^(n-1) = 524288)


In [22]:
# Elimination works over any field. Over GF(2) (1 + 1 = 0): x+y=1, y+z=1, x+y+z=1
def rref_gf2(M):
    M = np.array(M) % 2; m, n = M.shape; r = 0
    for c in range(n - 1):
        p = [i for i in range(r, m) if M[i, c]]
        if not p: continue
        M[[r, p[0]]] = M[[p[0], r]]
        for i in range(m):
            if i != r and M[i, c]: M[i] = (M[i] + M[r]) % 2
        r += 1
    return M, r
R2, rk = rref_gf2([[1, 1, 0, 1], [0, 1, 1, 1], [1, 1, 1, 1]]); print(R2, "rank", rk)
a = np.array([[1], [1]])
print("over R: rank(a^T a) =", np.linalg.matrix_rank(a.T @ a), "| over GF(2): a^T a mod 2 =", (a.T @ a) % 2, "-> rank 0, but rank a = 1")

[[1 0 0 0]
 [0 1 0 1]
 [0 0 1 0]] rank 3
over R: rank(a^T a) = 1 | over GF(2): a^T a mod 2 = [[0]] -> rank 0, but rank a = 1


## Part H — Checking the practice-problem answers

In [23]:
M = sp.Matrix
print("P1 :", M([[1, 1, 1], [2, 3, 1], [1, -1, 2]]).solve(M([6, 11, 5])).T)
print("P2 :", M([[1, 2, 3], [2, 4, 6], [1, 1, 1]]).rank())
print("P9 :", M([[2, 1], [5, 3]]).inv())
print("P10:", M([[1, 2], [3, 8]]).LUdecomposition()[:2], M([[1, 2], [3, 8]]).solve(M([5, 19])).T)
k, m = sp.symbols("k m"); R = M([[1, 1, 1, 1], [1, 2, 3, 2], [1, 3, k, m]])
R[1, :] -= R[0, :]; R[2, :] -= R[0, :]; R[2, :] -= 2 * R[1, :]; print("P11:", R.tolist())
print("P12:", M([[1, 2, 1, 0], [2, 4, 3, 1], [3, 6, 4, 1]]).rref())
print("P17:"); P, L, U = sl.lu(np.array([[1, 2, 1], [4, 1, 0], [2, 3, 5]], float)); print(P.T, L, U, sep="\n")
print("P19:", M([[1, 2, -1, 1, 2], [2, 4, 1, 5, 7]]).rref())
print("W5 :", M([[1, 2, -1], [2, 5, 1], [1, 3, 2]]).row_join(M([3, 8, 6])).rref()[0].tolist())
print("op counts  sum 2(n-k)^2 vs (n-1)n(2n-1)/3:", [(n, sum(2 * (n - j) ** 2 for j in range(1, n)), (n - 1) * n * (2 * n - 1) // 3) for n in (3, 10, 100)])

P1 : Matrix([[1, 2, 3]])
P2 : 2
P9 : Matrix([[3, -1], [-5, 2]])
P10: (Matrix([
[1, 0],
[3, 1]]), Matrix([
[1, 2],
[0, 2]])) Matrix([[1, 2]])
P11: [[1, 1, 1, 1], [0, 1, 2, 1], [0, 0, k - 5, m - 3]]
P12: (Matrix([
[1, 2, 0, -1],
[0, 0, 1,  1],
[0, 0, 0,  0]]), (0, 2))
P17:
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
[[1.   0.   0.  ]
 [0.5  1.   0.  ]
 [0.25 0.7  1.  ]]
[[ 4.   1.   0. ]
 [ 0.   2.5  5. ]
 [ 0.   0.  -2.5]]
P19: (Matrix([
[1, 2, 0, 2, 3],
[0, 0, 1, 1, 1]]), (0, 2))
W5 : [[1, 0, -7, 0], [0, 1, 3, 0], [0, 0, 0, 1]]
op counts  sum 2(n-k)^2 vs (n-1)n(2n-1)/3: [(3, 10, 10), (10, 570, 570), (100, 656700, 656700)]
